In [47]:
import tsim
import stim

In [48]:
p = 0.0001
Psi = stim.Circuit.generated(
    "surface_code:rotated_memory_x",
    rounds=1,
    distance=3,
    before_round_data_depolarization=p,
    before_measure_flip_probability=p,
    after_clifford_depolarization=p,
    after_reset_flip_probability=p,
)
Psi = tsim.Circuit.from_stim_program(Psi)


In [49]:
Psi.diagram(height=700)

In [50]:
def mappa_completa_surface_code(distance=3, rounds=2, p=0.001):
    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=rounds,
        distance=distance,
        before_round_data_depolarization=p,
        before_measure_flip_probability=p,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
    )

    n = circuit.num_qubits
    print(f"Totale qubit nel circuito: {n}")

    # --- 1. Coordinate (alcuni qubit potrebbero non averle) ---
    coords = {}
    for op in circuit:
        if op.name == "QUBIT_COORDS":
            q = op.targets_copy()[0].value
            coords[q] = tuple(op.gate_args_copy())

    # --- 2. Analizza istruzioni per classificare ---
    reset_r      = set()   # reset in |0⟩  → possibili ancilla Z
    reset_rx     = set()   # reset in |+⟩  → possibili ancilla X
    cnot_control = set()
    cnot_target  = set()
    measured     = set()

    for op in circuit:
        if op.name == "R":
            for t in op.targets_copy():
                reset_r.add(t.value)
        elif op.name == "RX":
            for t in op.targets_copy():
                reset_rx.add(t.value)
        elif op.name in ("CNOT", "CX"):
            t = op.targets_copy()
            for i in range(0, len(t), 2):
                cnot_control.add(t[i].value)
                cnot_target.add(t[i+1].value)
        elif op.name == "M":
            for t in op.targets_copy():
                measured.add(t.value)

    # --- 3. Classifica ogni qubit ---
    print(f"\n{'Idx':>4} | {'Ruolo':>10} | {'Coord':>12} | {'Note'}")
    print("-" * 50)

    ruoli = {}
    for q in range(n):
        coord_str = str(coords.get(q, "N/A"))

        # Data: compare sia come controllo che target in CNOT,
        #       e tipicamente NON viene resettato durante i round
        if q in cnot_control and q in cnot_target:
            role = "DATA"
        # Ancilla X: resettata in |+⟩, mai target di CNOT
        elif q in reset_rx and q not in cnot_target:
            role = "ANCILLA_X"
        # Ancilla Z: resettata in |0⟩, mai controllo di CNOT
        elif q in reset_r and q not in cnot_control:
            role = "ANCILLA_Z"
        else:
            role = "???"

        ruoli[q] = role
        note = ""
        if q not in coords:
            note = " (senza coordinate!)"
        print(f"{q:>4} | {role:>10} | {coord_str:>12} | {note}")

    
    return ruoli, coords, circuit

# Esegui
ruoli, coords, circuit = mappa_completa_surface_code(distance=3, rounds=2, p=0.001)

Totale qubit nel circuito: 26

 Idx |      Ruolo |        Coord | Note
--------------------------------------------------
   0 |        ??? |          N/A |  (senza coordinate!)
   1 |       DATA |   (1.0, 1.0) | 
   2 |        ??? |   (2.0, 0.0) | 
   3 |       DATA |   (3.0, 1.0) | 
   4 |        ??? |          N/A |  (senza coordinate!)
   5 |       DATA |   (5.0, 1.0) | 
   6 |        ??? |          N/A |  (senza coordinate!)
   7 |        ??? |          N/A |  (senza coordinate!)
   8 |       DATA |   (1.0, 3.0) | 
   9 |  ANCILLA_Z |   (2.0, 2.0) | 
  10 |       DATA |   (3.0, 3.0) | 
  11 |        ??? |   (4.0, 2.0) | 
  12 |       DATA |   (5.0, 3.0) | 
  13 |  ANCILLA_Z |   (6.0, 2.0) | 
  14 |  ANCILLA_Z |   (0.0, 4.0) | 
  15 |       DATA |   (1.0, 5.0) | 
  16 |        ??? |   (2.0, 4.0) | 
  17 |       DATA |   (3.0, 5.0) | 
  18 |  ANCILLA_Z |   (4.0, 4.0) | 
  19 |       DATA |   (5.0, 5.0) | 
  20 |        ??? |          N/A |  (senza coordinate!)
  21 |        ??? |   

In [87]:
from collections import defaultdict

def mappa_geometrica_surface_code(distance=3, rounds=2, p=0.001):
    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=rounds,
        distance=distance,
        before_round_data_depolarization=p,
        before_measure_flip_probability=p,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
    )

    # --- Estrai coordinate ---
    coords = {}
    for op in circuit:
        if op.name == "QUBIT_COORDS":
            q = op.targets_copy()[0].value
            coords[q] = tuple(op.gate_args_copy())

    # --- Classifica tramite CNOT ---
    controls, targets = set(), set()
    for op in circuit:
        if op.name in ("CNOT", "CX"):
            t = op.targets_copy()
            for i in range(0, len(t), 2):
                controls.add(t[i].value)
                targets.add(t[i+1].value)

    data      = controls & targets
    ancilla_x = controls - targets
    ancilla_z = targets - controls

    # --- Costruisci griglia ---
    xs = [int(c[0]) for c in coords.values()]
    ys = [int(c[1]) for c in coords.values()]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)

    griglia = {}
    for q, (x, y) in coords.items():
        ix, iy = int(x), int(y)
        if q in data:
            simbolo = "D"
        elif q in ancilla_x:
            simbolo = "X"
        else:
            simbolo = "Z"
        griglia[(ix, iy)] = (q, simbolo)

    # --- Stampa la mappa ---
    print("Mappa geometrica (y ↓, x →):")
    print("Legenda: D = DATA, X = Ancilla X (|+⟩), Z = Ancilla Z (|0⟩)")
    print("   " + " ".join(f"{x:2}" for x in range(min_x, max_x + 1)))
    for y in range(max_y, min_y - 1, -1):
        row = f"{y:2} "
        for x in range(min_x, max_x + 1):
            if (x, y) in griglia:
                q, s = griglia[(x, y)]
                row += f"{s:2} "
            else:
                row += " . "
        print(row)

    # --- Elenco ---
    print("\nElenco qubit (indice → ruolo):")
    linee = []
    for q in sorted(coords.keys()):
        if q in data:
            ruolo = "DATA"
        elif q in ancilla_x:
            ruolo = "ANCILLA_X"
        else:
            ruolo = "ANCILLA_Z"
        linee.append(f"q{q} {ruolo}")
        print(linee[-1])

    # ⭐ IMPORTANTE: aggiunto 'coords' al return
    return {
        "linee": linee,
        "griglia": griglia,
        "coords": coords,           # <-- serve per trovare le catene
        "data": sorted(data),
        "ancilla_x": sorted(ancilla_x),
        "ancilla_z": sorted(ancilla_z),
    }


def trova_catene_logiche(info, distance=3):
    """
    Dalle coordinate dei data qubit, estrae:
      - Z_CHAIN: data qubit su una colonna verticale (stessa x)
      - X_CHAIN: data qubit su una riga orizzontale (stessa y)
    
    Le catene sono ordinate in modo da essere usate direttamente
    nel CNOT-ladder (dal basso verso l'alto per Z, da sinistra a destra per X).
    """
    data = info["data"]
    coords = info["coords"]

    # Prendi solo i data qubit che hanno coordinate
    dc = {q: coords[q] for q in data if q in coords}

    # Raggruppa per x (colonne) e per y (righe)
    # Usiamo round a 1 decimale per robustezza sui float
    x_groups = defaultdict(list)  # x -> [(qubit, y), ...]
    y_groups = defaultdict(list)  # y -> [(qubit, x), ...]

    for q, (x, y) in dc.items():
        xr = round(float(x), 1)
        yr = round(float(y), 1)
        x_groups[xr].append((q, float(y)))
        y_groups[yr].append((q, float(x)))

    # Z_CHAIN: colonna con più qubit (stessa x), ordinati per y crescente
    z_items = max(x_groups.values(), key=lambda items: len(items))
    z_chain = [q for q, _ in sorted(z_items, key=lambda t: t[1])]

    # X_CHAIN: riga con più qubit (stessa y), ordinati per x crescente
    x_items = max(y_groups.values(), key=lambda items: len(items))
    x_chain = [q for q, _ in sorted(x_items, key=lambda t: t[1])]

    # Verifica
    assert len(z_chain) == distance, f"Z_CHAIN ha {len(z_chain)} qubit, attesi {distance}"
    assert len(x_chain) == distance, f"X_CHAIN ha {len(x_chain)} qubit, attesi {distance}"

    print(f"\nCatena Z_L (colonna, {len(z_chain)} qubit): {z_chain}")
    print(f"Catena X_L (riga,    {len(x_chain)} qubit): {x_chain}")

    return z_chain, x_chain


# =============================================================================
# USO
# =============================================================================
if __name__ == "__main__":
    info = mappa_geometrica_surface_code(distance=3, rounds=2, p=0.001)
    z_chain, x_chain = trova_catene_logiche(info, distance=3)
    
    # Ora puoi usarle direttamente:
    print(f"\nEsempio ladder Z: CX {z_chain[0]} {z_chain[1]} ; CX {z_chain[1]} {z_chain[2]}")

Mappa geometrica (y ↓, x →):
Legenda: D = DATA, X = Ancilla X (|+⟩), Z = Ancilla Z (|0⟩)
    0  1  2  3  4  5  6
 6  .  .  .  . X   .  . 
 5  . D   . D   . D   . 
 4 Z   . X   . Z   .  . 
 3  . D   . D   . D   . 
 2  .  . Z   . X   . Z  
 1  . D   . D   . D   . 
 0  .  . X   .  .  .  . 

Elenco qubit (indice → ruolo):
q1 DATA
q2 ANCILLA_X
q3 DATA
q5 DATA
q8 DATA
q9 ANCILLA_Z
q10 DATA
q11 ANCILLA_X
q12 DATA
q13 ANCILLA_Z
q14 ANCILLA_Z
q15 DATA
q16 ANCILLA_X
q17 DATA
q18 ANCILLA_Z
q19 DATA
q25 ANCILLA_X

Catena Z_L (colonna, 3 qubit): [1, 8, 15]
Catena X_L (riga,    3 qubit): [1, 3, 5]

Esempio ladder Z: CX 1 8 ; CX 8 15


In [ ]:

def logical_rz(theta: float, z_chain: list) -> tsim.Circuit:
    """
    Applica R_Z(theta) logico tramite CNOT-ladder su una catena di data qubit.
    z_chain: lista ordinata di qubit fisici che formano Z_L (es. [0, 3, 6]).
    """
    ops = []
    
    # Forward: comprimi Z_L sull'ultimo qubit della catena
    for i in range(len(z_chain) - 1):
        ops.append(f"CX {z_chain[i]} {z_chain[i+1]}")
    
    # Rotazione fisica sul target (tsim supporta R_Z nativamente)
    ops.append(f"R_Z({theta}) {z_chain[-1]}")
    
    # Backward: decomprimi
    for i in range(len(z_chain) - 2, -1, -1):
        ops.append(f"CX {z_chain[i]} {z_chain[i+1]}")
    
    ops.append("TICK")
    return tsim.Circuit("\n".join(ops))



# Esempio con il tuo layout
Z_CHAIN = z_chain   # dal tuo surface code d=3                # qubit ausiliario (fuori dal patch)
print (z_chain)
theta = 0,001  # Cambia qui: 0, 1.57 (π/2), 3.14 (π), etc.
# Riesegui tutto
gate_cz = logical_rz(theta, Z_CHAIN)
gate_cz_circ = tsim.Circuit.from_stim_program(gate_cz)


[1, 8, 15]


In [ ]:
import tsim
import stim
import numpy as np

p = 0.0001
distance = 3
rounds = 1
theta = 0.001  

# --- 1. Genera circuito base ---
base = stim.Circuit.generated(
    "surface_code:rotated_memory_x",
    rounds=rounds,
    distance=distance,
    before_round_data_depolarization=p,
    before_measure_flip_probability=p,
    after_clifford_depolarization=p,
    after_reset_flip_probability=p,
)

# --- 2. Estrai catene dal tuo layout ---
info = mappa_geometrica_surface_code(distance=distance, rounds=rounds, p=p)
z_chain, x_chain = trova_catene_logiche(info, distance=distance)

data_qubits = sorted(info["data"])
print(f"Z_CHAIN: {z_chain}")
print(f"X_CHAIN: {x_chain}")
print(f"Data qubits: {data_qubits}")

# --- 3. Togli l'ultimo layer di misura (MX + DETECTOR + OBSERVABLE) ---
lines = str(base).strip().split('\n')
prep_lines = []

for line in lines:
    s = line.strip()
    # Fermati appena incontri MX, DETECTOR o OBSERVABLE
    if s.startswith('MX') or s.startswith('DETECTOR') or s.startswith('OBSERVABLE'):
        break
    prep_lines.append(line)

# --- 4. Costruisci il gate logico R_Z(theta) ---
gate_lines = []
for i in range(len(z_chain) - 1):
    gate_lines.append(f"CX {z_chain[i]} {z_chain[i+1]}")
gate_lines.append(f"R_Z({theta}) {z_chain[-1]}")
for i in range(len(z_chain) - 2, -1, -1):
    gate_lines.append(f"CX {z_chain[i]} {z_chain[i+1]}")

# --- 5. Rimetti le MX finali su TUTTI i data qubit ---
mx_lines = [f"MX {q}" for q in data_qubits]

# --- 6. Assembla in tsim ---
full_str = '\n'.join(prep_lines + gate_lines + mx_lines)
circuito = tsim.Circuit(full_str)
circuito.diagram(height=700)




Mappa geometrica (y ↓, x →):
Legenda: D = DATA, X = Ancilla X (|+⟩), Z = Ancilla Z (|0⟩)
    0  1  2  3  4  5  6
 6  .  .  .  . X   .  . 
 5  . D   . D   . D   . 
 4 Z   . X   . Z   .  . 
 3  . D   . D   . D   . 
 2  .  . Z   . X   . Z  
 1  . D   . D   . D   . 
 0  .  . X   .  .  .  . 

Elenco qubit (indice → ruolo):
q1 DATA
q2 ANCILLA_X
q3 DATA
q5 DATA
q8 DATA
q9 ANCILLA_Z
q10 DATA
q11 ANCILLA_X
q12 DATA
q13 ANCILLA_Z
q14 ANCILLA_Z
q15 DATA
q16 ANCILLA_X
q17 DATA
q18 ANCILLA_Z
q19 DATA
q25 ANCILLA_X

Catena Z_L (colonna, 3 qubit): [1, 8, 15]
Catena X_L (riga,    3 qubit): [1, 3, 5]
Z_CHAIN: [1, 8, 15]
X_CHAIN: [1, 3, 5]
Data qubits: [1, 3, 5, 8, 10, 12, 15, 17, 19]


In [105]:
# --- 7. Simula ---
sampler = circuito.compile_sampler()
shots = sampler.sample(shots=100_000)

# --- 8. Post-selection: XOR delle MX su x_chain ---
# Le ultime len(data_qubits) colonne sono le MX finali
num_data = len(data_qubits)
mx_bits = shots[:, -num_data:]

# Mappa qubit -> colonna relativa nel blocco MX
qubit_to_col = {q: i for i, q in enumerate(data_qubits)}
x_cols = [qubit_to_col[q] for q in x_chain]

# XOR (parità) = outcome logico di X_L
logical_x = np.mod(np.sum(mx_bits[:, x_cols], axis=1), 2)

# Tieni solo outcome 1 (|-⟩_L), scarta 0 (|+⟩_L)
selected = shots[logical_x == 1]
discarded = shots[logical_x == 0]

print(f"\nShot totali: {len(shots)}")
print(f"Selezionati (outcome 1, |-⟩_L): {len(selected)} ({len(selected)/len(shots):.4f})")
print(f"Scartati (outcome 0, |+⟩_L): {len(discarded)} ({len(discarded)/len(shots):.4f})")



Shot totali: 100000
Selezionati (outcome 1, |-⟩_L): 50297 (0.5030)
Scartati (outcome 0, |+⟩_L): 49703 (0.4970)
